# Chapitre 18 - Problèmes en grande dimension ($p \gg N$)
## Cours 2 : régularisation quadratique, astuce du noyau, pénalités L1, classification sans variables

Plan (sections 18.3 à 18.5 du livre) :

1. classifieurs linéaires à **pénalité quadratique** (RDA, logistique ridge, SVM) ;
2. l'**astuce de calcul** : quand $p > N$, tout se ramène à un problème en dimension
   $N$ grâce à la SVD (coût $O(N^2p)$ au lieu de $O(p^3)$) ;
3. pénalités **L1** : lasso, elastic net, *fused lasso* ;
4. classification quand on ne dispose que de **produits scalaires** ou de
   **distances** (noyaux de chaînes, centroïdes, ACP à noyau).

In [1]:
import time
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
set_style()

from esl import datasets
from esl import ch18_grande_dimension as g

## 1. Classifieurs linéaires à pénalité quadratique

### 1.1 Analyse discriminante régularisée (RDA)

La LDA utilise l'inverse de la matrice de covariance intra-classe $\hat\Sigma$
($p\times p$). Quand $p \gg N$, $\hat\Sigma$ est de rang au plus $N - K$ : elle est
singulière. La RDA la remplace par un mélange avec sa diagonale (éq. (18.9) du livre) :
$$\hat\Sigma(\gamma) = \gamma\,\hat\Sigma + (1-\gamma)\operatorname{diag}(\hat\Sigma), \qquad \gamma\in[0,1].$$
Pour $\gamma < 1$, $\hat\Sigma(\gamma)$ est définie positive (somme d'une matrice
positive et d'une matrice diagonale à termes $> 0$), donc inversible.
$\gamma = 0$ redonne la **LDA diagonale** du cours 1 ; $\gamma \to 1$ la LDA complète.

> **Intuition.** C'est l'analogue de la ridge : on « tire » la covariance estimée
> vers une matrice diagonale, ce qui stabilise son inversion. L'exercice 18.6 montre
> que le calcul se fait, lui aussi, en dimension $N$.

### 1.2 Régression logistique multinomiale ridge

Avec $K$ classes, on utilise la forme **symétrique** du modèle logistique
(éq. (18.10)) :
$$\Pr(G = k\mid X = x) = \frac{\exp(\beta_{k0} + x^T\beta_k)}{\sum_{\ell=1}^K \exp(\beta_{\ell 0} + x^T\beta_\ell)},$$
avec un vecteur $\beta_k\in\mathbb R^p$ **par classe**. Sans pénalité ce modèle est
sur-paramétré (ajouter le même vecteur à tous les $\beta_k$ ne change pas les
probabilités). On maximise la log-vraisemblance pénalisée (éq. (18.11)) :
$$\max_{\{\beta_{k0},\beta_k\}}\ \sum_{i=1}^N \log\Pr(g_i\mid x_i) - \frac\lambda2\sum_{k=1}^K\|\beta_k\|_2^2 .$$
La pénalité lève l'indétermination et impose $\sum_k\hat\beta_{kj} = 0$ pour chaque
$j$ (exercice 18.3). Le problème est **concave** (log-vraisemblance concave moins
une fonction convexe) : un algorithme de type Newton ou quasi-Newton le résout.

### 1.3 Le classifieur à vecteurs supports (SVM)

Quand $p > N$, deux classes sont en général **linéairement séparables** (sauf si deux
observations identiques ont des classes différentes) : $N$ points en dimension
$p \ge N$ peuvent recevoir n'importe quelles valeurs $x_i^T\beta$. Le SVM sans
régularisation trouve alors l'hyperplan séparateur de **marge maximale**
(section 12.2). Le livre note qu'en pratique, quand $p\gg N$, ce SVM « dur »
fonctionne souvent aussi bien que ses versions régularisées.

## 2. L'astuce de calcul quand $p > N$ (section 18.3.5)

**Idée géométrique** : $N$ points de $\mathbb R^p$ vivent dans un sous-espace de
dimension au plus $N$ (deux points de $\mathbb R^3$ sont toujours sur une droite).
On peut donc tourner les axes pour que seules $N$ coordonnées soient non nulles.

On écrit la SVD réduite (éq. (18.12)-(18.13)) :
$$\mathbf X = \mathbf U\mathbf D\mathbf V^T = \mathbf R\mathbf V^T,\qquad \mathbf R = \mathbf U\mathbf D\ (N\times N),$$
où $\mathbf V$ ($p\times N$) a des colonnes orthonormées ($\mathbf V^T\mathbf V = I_N$)
et $\mathbf U$ est $N\times N$ orthogonale. La ligne $i$ de $\mathbf R$, notée
$r_i^T$, donne les **coordonnées** de $x_i$ dans la base des colonnes de $\mathbf V$ :
$x_i = \mathbf V r_i$.

**Ridge en dimension $N$.** La solution ridge (18.14)
$\hat\beta = (\mathbf X^T\mathbf X+\lambda I)^{-1}\mathbf X^Ty$ s'écrit aussi
(éq. (18.15), démontrée à l'exercice 18.4) :
$$\hat\beta = \mathbf V(\mathbf R^T\mathbf R + \lambda I_N)^{-1}\mathbf R^Ty .$$
Autrement dit $\hat\beta = \mathbf V\hat\theta$, où $\hat\theta$ est la ridge calculée
sur les $N$ observations $(r_i, y_i)$ à $N$ variables.

**Coût.** La SVD d'une matrice $N\times p$ coûte $O(N^2p)$ ; il reste ensuite à
résoudre un système $N\times N$. Le calcul direct résout un système $p\times p$ :
$O(p^3)$. Pour $p = 6830$ et $N = 57$, le rapport est de l'ordre de
$p^2/N^2 \approx 1{,}4\cdot 10^4$.

Vérifions numériquement l'égalité des deux formules.

In [ ]:
rng = np.random.RandomState(0)
N, p = 50, 2000
Xs = rng.randn(N, p)
ys = Xs[:, :10].sum(1) + rng.randn(N)
b_direct = g.ridge_direct(Xs, ys, 5.0)
b_svd = g.ridge_svd(Xs, ys, 5.0)
print("écart max entre les deux formules :", np.max(np.abs(b_direct - b_svd)))

Comparons les temps de calcul quand $p$ augmente ($N = 50$ fixé).

In [ ]:
def best_time(f, n_rep=3):
    """Meilleur temps sur n_rep exécutions (moins sensible à la charge de la machine)."""
    best = np.inf
    for _ in range(n_rep):
        t0 = time.perf_counter()
        f()
        best = min(best, time.perf_counter() - t0)
    return best


p_list = [500, 1000, 2000, 4000]
t_dir, t_svd = [], []
for p in p_list:
    Xp = rng.randn(50, p)
    yp = rng.randn(50)
    t_dir.append(best_time(lambda: g.ridge_direct(Xp, yp, 1.0)))
    t_svd.append(best_time(lambda: g.ridge_svd(Xp, yp, 1.0)))
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(p_list, t_dir, "o-", color=ORANGE, label="formule directe (système p × p)")
ax.loglog(p_list, t_svd, "o-", color=BLUE, label="via la SVD (18.15)")
pp = np.array(p_list, dtype=float)
ax.loglog(pp, t_dir[-1] * (pp / pp[-1]) ** 3, ":", color=GRAY, lw=1, label="pente 3")
ax.loglog(pp, t_svd[-1] * (pp / pp[-1]), "--", color=GRAY, lw=1, label="pente 1")
ax.set_xlabel("nombre de variables p")
ax.set_ylabel("temps de calcul (s)")
ax.set_title("Ridge avec N = 50 : coût O(p³) contre O(N²p)")
ax.legend()
plt.show()

La formule directe est toujours bien plus lente, et l'écart se creuse quand $p$
augmente : en théorie, sur une échelle log-log, la pente vaut 3 pour le calcul direct
(coût cubique) et 1 pour la SVD (coût linéaire en $p$), voir les droites grises. Les
temps mesurés dépendent de la machine, de sa charge et de la bibliothèque d'algèbre
linéaire (parallélisme) : seule la tendance compte.

### Le théorème général (éq. (18.16)-(18.17))

Le résultat vaut pour **tout modèle linéaire à pénalité quadratique**. Soit une perte
$L$ quelconque (moins la log-vraisemblance logistique, par exemple) et les deux
problèmes
$$\begin{aligned}
(\hat\beta_0,\hat\beta) &= \arg\min_{\beta_0,\ \beta\in\mathbb R^p}\ \sum_{i=1}^N L(y_i, \beta_0 + x_i^T\beta) + \lambda\,\beta^T\beta ,\\
(\hat\theta_0,\hat\theta) &= \arg\min_{\theta_0,\ \theta\in\mathbb R^N}\ \sum_{i=1}^N L(y_i, \theta_0 + r_i^T\theta) + \lambda\,\theta^T\theta .
\end{aligned}$$
Alors $\hat\beta_0 = \hat\theta_0$ et $\hat\beta = \mathbf V\hat\theta$.

**Idée de la preuve** (détaillée à l'exercice 18.5). On décompose
$\beta = \mathbf V\theta + \beta_\perp$ avec $\mathbf V^T\beta_\perp = 0$. Alors
$x_i^T\beta = r_i^T\mathbf V^T\mathbf V\theta + r_i^T\mathbf V^T\beta_\perp = r_i^T\theta$ :
la perte ne dépend pas de $\beta_\perp$. Par Pythagore,
$\|\beta\|^2 = \|\theta\|^2 + \|\beta_\perp\|^2$ : la pénalité est minimale pour
$\beta_\perp = 0$. Ce n'est vrai que parce que la pénalité est **invariante par
rotation** - ce qui n'est pas le cas de la pénalité L1 du lasso.

**Vérification sur une logistique ridge** ($N = 40$, $p = 1000$). On ajuste le
modèle (par Newton-Raphson) une fois avec les $x_i \in\mathbb R^{1000}$, une fois
avec les $r_i\in\mathbb R^{40}$.

In [ ]:
rng = np.random.RandomState(1)
Xl = rng.randn(40, 1000)
yl = (Xl[:, :5].sum(1) + rng.randn(40) > 0).astype(float)
t0 = time.perf_counter()
b0_full, b_full = g.ridge_logistic(Xl, yl, lam=2.0)
t_full = time.perf_counter() - t0
t0 = time.perf_counter()
R, V = g.svd_reduce(Xl)
th0, th = g.ridge_logistic(R, yl, lam=2.0)
t_red = time.perf_counter() - t0
print(f"|beta0 - theta0|        = {abs(b0_full - th0):.2e}")
print(f"max |beta - V theta|    = {np.max(np.abs(b_full - V @ th)):.2e}")
print(f"temps : p = 1000 -> {t_full:.2f} s ; réduit (N = 40) -> {t_red:.3f} s")